In [1]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent

sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT:", PROJECT_ROOT)
print("sys.path[0]:", sys.path[0])


PROJECT_ROOT: /data/ceph/hdd/project/node_09/sys_gen_students/2025_2026/p06_indel_glm/repo_jun/InsDelGLM
sys.path[0]: /data/ceph/hdd/project/node_09/sys_gen_students/2025_2026/p06_indel_glm/repo_jun/InsDelGLM


In [2]:
from transformers import BertConfig
from modules.custom_bert import DNABertForMaskedLM
from modules.dna_tokenizer import DNATokenizer
import torch

model_dir = "../model/test"

tokenizer = DNATokenizer.from_pretrained(model_dir)
model = DNABertForMaskedLM.from_pretrained(model_dir)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

/opt/modules/i12g/anaconda/envs/sysgen_p06/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DNABertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(9, 256, padding_idx=0)
      (position_embeddings): Embedding(512, 256)
      (token_type_embeddings): Embedding(1, 256)
      (LayerNorm): LayerNorm((256,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-5): 6 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=256, out_features=256, bias=True)
              (key): Linear(in_features=256, out_features=256, bias=True)
              (value): Linear(in_features=256, out_features=256, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=256, out_features=256, bias=True)
              (LayerNorm): LayerNorm((256,), eps=1e-12, elementwise_

In [5]:
!ls -lh ../model/test

total 19M
-rw-rw----+ 1 s_jkan root  580 Dec 29 11:28 config.json
-rw-rw----+ 1 s_jkan root  19M Dec 29 11:28 model.safetensors
-rw-rw----+ 1 s_jkan root    3 Dec 29 11:28 special_tokens_map.json
-rw-rw----+ 1 s_jkan root  294 Dec 29 11:28 tokenizer_config.json
-rw-rw----+ 1 s_jkan root 5.4K Dec 29 11:28 training_args.bin
-rw-rw----+ 1 s_jkan root   35 Dec 29 11:28 vocab.txt


In [3]:
import pandas as pd

test_path = "../data/simulated/test.parquet"
test_df = pd.read_parquet(test_path)

print("n_test:", len(test_df))
print(test_df.head(2))
print("example len:", len(test_df["sequences"].iloc[0]))
print("gap count example:", test_df["sequences"].iloc[0].count("-"))

n_test: 1000
                                              sequences labels    id
2487  CTCTGGACAAGAGTTGAGCACAATCGTGTGA-TGACCCAAT-TG--...      A  2487
1264  CCACGAC-ACGACCGTTACCGGCC-ACGTGATCGCCAAATCAT-A-...   both  1264
example len: 512
gap count example: 25


In [4]:
import torch
from torch.utils.data import Dataset, DataLoader

class DNADataset(Dataset):
    def __init__(self, sequences, tokenizer, max_length=512):
        self.sequences = sequences
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.sequences[idx],
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {k: v.squeeze(0) for k, v in enc.items()}

test_seqs = test_df["sequences"].tolist()
test_ds = DNADataset(test_seqs, tokenizer, max_length=512)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

batch = next(iter(test_loader))
print({k: v.shape for k, v in batch.items()})

{'input_ids': torch.Size([16, 512]), 'attention_mask': torch.Size([16, 512])}


In [12]:
import torch
import torch.nn.functional as F
import numpy as np
import math

mask_id = tokenizer.mask_token_id
pad_id  = tokenizer.pad_token_id

DNA_TOKEN_IDS = [
    tokenizer("A")["input_ids"][0][1],
    tokenizer("C")["input_ids"][0][1],
    tokenizer("G")["input_ids"][0][1],
    tokenizer("T")["input_ids"][0][1],
    tokenizer("-")["input_ids"][0][1],
]

def mlm_eval_loss(model, batch, mlm_prob=0.15):
    input_ids = batch["input_ids"].clone().to(device)
    attn_mask = batch["attention_mask"].clone().to(device)

    labels = input_ids.clone()

    # mask positions
    special = (input_ids == pad_id)
    rand = torch.rand(input_ids.shape, device=device)
    mask_positions = (rand < mlm_prob) & (~special) & (attn_mask == 1)

    labels[~mask_positions] = -100

    # create corrupted inputs (80/10/10)
    prob = torch.rand(input_ids.shape, device=device)
    input_ids[mask_positions & (prob < 0.8)] = mask_id

    rand_pos = mask_positions & (prob >= 0.8) & (prob < 0.9)
    if rand_pos.any():
        r = torch.randint(low=0, high=len(DNA_TOKEN_IDS), size=input_ids.shape, device=device)
        rand_tokens = torch.tensor(DNA_TOKEN_IDS, device=device)[r]
        input_ids[rand_pos] = rand_tokens[rand_pos]

    with torch.no_grad():
        out = model(input_ids=input_ids, attention_mask=attn_mask)

    # out can be tuple or object; extract logits robustly
    logits = out.logits if hasattr(out, "logits") else out[0]   # [B, L, V]

    # compute CE only where labels != -100
    loss = F.cross_entropy(
        logits.view(-1, logits.size(-1)),
        labels.view(-1),
        ignore_index=-100,
        reduction="mean",
    )
    return float(loss.item())

losses = [mlm_eval_loss(model, batch, mlm_prob=0.15) for batch in test_loader]
mean_loss = float(np.mean(losses))
ppl = float(math.exp(mean_loss))

print("TEST MLM loss:", mean_loss)
print("TEST perplexity:", ppl)

TEST MLM loss: 1.505757210746644
TEST perplexity: 4.5075655154168945


In [5]:
import numpy as np
import torch
import torch.nn.functional as F

gap_id = tokenizer("-")["input_ids"][0][1]
mask_id = tokenizer.mask_token_id
pad_id  = tokenizer.pad_token_id

DNA_TOKEN_IDS = [
    tokenizer("A")["input_ids"][0][1],
    tokenizer("C")["input_ids"][0][1],
    tokenizer("G")["input_ids"][0][1],
    tokenizer("T")["input_ids"][0][1],
    gap_id,
]

print("gap_id:", int(gap_id), "mask_id:", int(mask_id), "pad_id:", int(pad_id))
print("DNA_TOKEN_IDS:", [int(x) for x in DNA_TOKEN_IDS])

gap_id: 4 mask_id: 6 pad_id: 5
DNA_TOKEN_IDS: [0, 2, 3, 1, 4]


In [6]:
def eval_gap_vs_nongap_nll(model, batch, mlm_prob=0.15):
    input_ids = batch["input_ids"].clone().to(device)
    attn_mask = batch["attention_mask"].clone().to(device)

    labels = input_ids.clone()

    special = (input_ids == pad_id)

    rand = torch.rand(input_ids.shape, device=device)
    mask_positions = (rand < mlm_prob) & (~special) & (attn_mask == 1)

    labels[~mask_positions] = -100

    gap_mask = mask_positions & (input_ids == gap_id)
    nongap_mask = mask_positions & (input_ids != gap_id)

    prob = torch.rand(input_ids.shape, device=device)
    input_ids[mask_positions & (prob < 0.8)] = mask_id

    rand_pos = mask_positions & (prob >= 0.8) & (prob < 0.9)
    if rand_pos.any():
        r = torch.randint(0, len(DNA_TOKEN_IDS), size=input_ids.shape, device=device)
        rand_tokens = torch.tensor(DNA_TOKEN_IDS, device=device)[r]
        input_ids[rand_pos] = rand_tokens[rand_pos]

    with torch.no_grad():
        out = model(input_ids=input_ids, attention_mask=attn_mask)
    logits = out.logits if hasattr(out, "logits") else out[0]

    # token-level negative log likelihood
    log_probs = torch.log_softmax(logits, dim=-1)

    def mean_nll(mask):
        if not mask.any():
            return None
        gold = labels[mask] 
        lp = log_probs[mask, :].gather(1, gold.unsqueeze(1)).squeeze(1)  # 정답 logprob
        return float((-lp).mean().item())

    return mean_nll(gap_mask), mean_nll(nongap_mask)

In [15]:
torch.manual_seed(0)

gap_nlls = []
nongap_nlls = []

for batch in test_loader:
    g, ng = eval_gap_vs_nongap_nll(model, batch, mlm_prob=0.15)
    if g is not None:
        gap_nlls.append(g)
    if ng is not None:
        nongap_nlls.append(ng)

print("TEST gap NLL (lower=better):", float(np.mean(gap_nlls)) if gap_nlls else None)
print("TEST non-gap NLL:", float(np.mean(nongap_nlls)) if nongap_nlls else None)
print("gap/non-gap ratio:", float(np.mean(gap_nlls)/np.mean(nongap_nlls)) if gap_nlls and nongap_nlls else None)

TEST gap NLL (lower=better): 2.737488867744567
TEST non-gap NLL: 1.439737666220892
gap/non-gap ratio: 1.901380322243071


In [16]:
!pwd

/data/ceph/hdd/project/node_09/sys_gen_students/2025_2026/p06_indel_glm/repo_jun/InsDelGLM/notebooks


In [31]:
import numpy as np
import torch
import pandas as pd

# device
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# load motifs
with open("../data/simulated/motifs.txt") as f:
    motif_a = f.readline().strip()
    motif_b = f.readline().strip()

lb = len(motif_b)
print("motif_b:", motif_b, "len:", lb)

mask_id = tokenizer.mask_token_id
pad_id  = tokenizer.pad_token_id

motif_b: TTTGAG len: 6


In [11]:
def char_id(ch: str) -> int:
    out = tokenizer(ch)  # usually returns [CLS, ch, SEP, PAD...]
    ids = out["input_ids"]
    if hasattr(ids, "tolist"):
        ids = ids.squeeze(0).tolist()
    # [CLS, ch, SEP] 형태면 ch는 index 1
    return ids[1] if len(ids) >= 3 else ids[0]

VOCAB = {ch: char_id(ch) for ch in ["A","C","G","T","-"]}
print("VOCAB:", VOCAB)

VOCAB: {'A': 0, 'C': 2, 'G': 3, 'T': 1, '-': 4}


In [12]:
def avg_logprob_masked_motif(sequence: str, motif: str, start: int, max_length=512):
   
    if start < 0 or start + len(motif) > len(sequence):
        return None

    enc = tokenizer(sequence, return_tensors="pt", padding="max_length", max_length=max_length)
    input_ids = enc["input_ids"].to(device)       # [1,L]
    attn_mask = enc["attention_mask"].to(device)  # [1,L]

    offset = 1
    pos_list = [start + j + offset for j in range(len(motif))]

    L = input_ids.shape[1]
    if any(p >= L for p in pos_list):
        return None

    masked_ids = input_ids.clone()
    masked_ids[0, pos_list] = mask_id

    with torch.no_grad():
        out = model(input_ids=masked_ids, attention_mask=attn_mask)

    logits = out.logits if hasattr(out, "logits") else out[0]  # [1,L,V]
    logits = logits[0]  # [L,V]
    logp = torch.log_softmax(logits, dim=-1)

    lps = []
    for j, ch in enumerate(motif):
        p = pos_list[j]
        ch_tok = VOCAB.get(ch, None)
        if ch_tok is None:
            return None
        lps.append(logp[p, ch_tok].item())

    return float(np.mean(lps))

In [13]:
def find_motif_pos(seq: str, motif: str) -> int:
    return seq.find(motif)

def overwrite_str(s: str, start: int, sub: str) -> str:
    s = list(s)
    for i, ch in enumerate(sub):
        if start + i < len(s):
            s[start + i] = ch
    return "".join(s)

def find_motif_pos(seq: str, motif: str) -> int:
    return seq.find(motif)

def overwrite_str(s: str, start: int, sub: str) -> str:
    s = list(s)
    for i, ch in enumerate(sub):
        if start + i < len(s):
            s[start + i] = ch
    return "".join(s)

def shift_delta_from_seq(seq: str, shift: int):
    """
    Δ = score_valid - score_shift
    
    """
    b_start = find_motif_pos(seq, motif_b)
    if b_start < 0:
        return None

    b2 = b_start + shift
    if b2 < 0 or b2 + lb > len(seq):
        return None

    seq_valid = overwrite_str(seq, b_start, motif_b)
    seq_shift = overwrite_str(seq, b2, motif_b)

    s_valid = avg_logprob_masked_motif(seq_valid, motif_b, b_start)
    s_shift = avg_logprob_masked_motif(seq_shift, motif_b, b2)

    if s_valid is None or s_shift is None:
        return None

    return s_valid - s_shift

In [16]:
df = pd.read_parquet("../data/simulated/test.parquet")
seqs = df["sequences"].tolist()
print("n_test:", len(seqs))

def collect_shift_deltas(seqs, n=1000, shift=3, seed=0):
    rng = np.random.default_rng(seed)
    deltas = []
    tries = 0
    while len(deltas) < n and tries < n * 50:
        tries += 1
        seq = seqs[rng.integers(0, len(seqs))]
        d = shift_delta_from_seq(seq, shift=shift)
        if d is None:
            continue
        deltas.append(d)
    return np.array(deltas)

for sh in [-7, -5, -3, -1, 1, 3, 5, 7]:
    d = collect_shift_deltas(seqs, n=1000, shift=sh, seed=0)
    print(
        f"shift={sh} | n={len(d)} | meanΔ={d.mean():.6f} | "
        f"medianΔ={np.median(d):.6f} | frac(Δ>0)={(d>0).mean():.3f}"
    )

n_test: 1000
shift=-7 | n=1000 | meanΔ=-0.000202 | medianΔ=-0.000214 | frac(Δ>0)=0.486
shift=-5 | n=1000 | meanΔ=-0.000505 | medianΔ=-0.000371 | frac(Δ>0)=0.490
shift=-3 | n=1000 | meanΔ=-0.000999 | medianΔ=0.000028 | frac(Δ>0)=0.503
shift=-1 | n=1000 | meanΔ=-0.000398 | medianΔ=-0.000678 | frac(Δ>0)=0.466
shift=1 | n=1000 | meanΔ=-0.001201 | medianΔ=-0.000935 | frac(Δ>0)=0.460
shift=3 | n=1000 | meanΔ=-0.000243 | medianΔ=-0.001209 | frac(Δ>0)=0.456
shift=5 | n=1000 | meanΔ=-0.000057 | medianΔ=-0.000636 | frac(Δ>0)=0.477
shift=7 | n=1000 | meanΔ=0.000742 | medianΔ=0.001020 | frac(Δ>0)=0.543


In [14]:
import numpy as np
import random

# motif 정보
with open("../data/simulated/motifs.txt") as f:
    motif_a = f.readline().strip()
    motif_b = f.readline().strip()

lb = len(motif_b)
DNA = ["A", "C", "G", "T"]

def random_dna(length):
    return "".join(random.choices(DNA, k=length))


def find_motif_pos(seq: str, motif: str) -> int:
    return seq.find(motif)


def overwrite_str(s: str, start: int, sub: str) -> str:
    s = list(s)
    for i, ch in enumerate(sub):
        if start + i < len(s):
            s[start + i] = ch
    return "".join(s)


def motif_vs_random_delta(seq: str):
    """
    Δ = logP(motif B | context) - logP(random DNA | context)
    evaluated at the SAME position with SAME surrounding context
    """
    b_start = find_motif_pos(seq, motif_b)
    if b_start < 0:
        return None

    # valid: motif B at original position
    seq_motif = overwrite_str(seq, b_start, motif_b)

    # control: random DNA (same length) at the same position
    rand_sub = random_dna(lb)
    seq_random = overwrite_str(seq, b_start, rand_sub)

    s_motif = avg_logprob_masked_motif(seq_motif, motif_b, b_start)
    s_random = avg_logprob_masked_motif(seq_random, rand_sub, b_start)

    if s_motif is None or s_random is None:
        return None

    return s_motif - s_random

In [15]:
import pandas as pd

df = pd.read_parquet("../data/simulated/test.parquet")
seqs = df["sequences"].tolist()

def collect_motif_vs_random_deltas(seqs, n=1000, seed=0):
    rng = np.random.default_rng(seed)
    deltas = []
    tries = 0

    while len(deltas) < n and tries < n * 50:
        tries += 1
        seq = seqs[rng.integers(0, len(seqs))]
        d = motif_vs_random_delta(seq)
        if d is not None:
            deltas.append(d)

    return np.array(deltas)


d = collect_motif_vs_random_deltas(seqs, n=1000, seed=0)

print("Motif vs Random Evaluation")
print("n:", len(d))
print("mean Δ:", float(d.mean()))
print("median Δ:", float(np.median(d)))
print("frac(Δ > 0):", float((d > 0).mean()))

Motif vs Random Evaluation
n: 1000
mean Δ: -0.0036157993078231856
median Δ: -0.004179785648981804
frac(Δ > 0): 0.389
